In [1]:
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import os
import re
import shutil
import csv
from google.colab import files

# --- 1. CARREGAR ARQUIVOS ---
print("1. Selecione a PLANILHA DA EQUIPE:")
up_p = files.upload()
nome_p = list(up_p.keys())[0]

print("\n2. Selecione o MODELO DA IMAGEM:")
up_i = files.upload()
nome_i = list(up_i.keys())[0]

# --- 2. CONFIGURAÇÕES DE DESIGN ---
ALTURA_INICIAL = 820  # Texto abaixo do logo
TAMANHO_FONTE = 42
ESPACO_LINHAS = 65
LARGURA_MAXIMA = 1450 # Define a largura do bloco de texto

pasta_saida = 'CERTIFICADOS_EQUIPE_SUCESSO'
if os.path.exists(pasta_saida):
    shutil.rmtree(pasta_saida)
os.makedirs(pasta_saida, exist_ok=True)

# Função para limpar caracteres nulos e invisíveis
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).replace('\x00', '')
    return "".join(char for char in text if char.isprintable()).strip()

# Função para desenhar alinhado à esquerda
def draw_left_aligned_line(draw, segments, y, margin_left):
    current_x = margin_left
    for text, font in segments:
        draw.text((current_x, y), text, fill=(30, 30, 30), font=font)
        current_x += draw.textlength(text, font=font)

# --- 3. PROCESSAMENTO ---
try:
    # Fontes oficiais
    f_reg = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", TAMANHO_FONTE)
    f_bold = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", TAMANHO_FONTE)

    # Leitura robusta da planilha
    df = None
    for enc in ['utf-8-sig', 'latin1', 'cp1252']:
        try:
            df = pd.read_csv(nome_p, sep=None, engine='python', encoding=enc)
            if df.shape[1] > 1: break
        except: continue

    if df is None:
        raise Exception("Não foi possível ler a planilha.")

    # Limpeza profunda contra erros de 'null byte'
    df = df.applymap(clean_text)

    print(f"\n⏳ Gerando certificados para a Equipe...")

    for index, row in df.iterrows():
        img = Image.open(nome_i).convert('RGB')
        draw = ImageDraw.Draw(img)
        L, A = img.size

        # Dados da equipe (Mapeamento por nome de coluna ou posição)
        nome = str(row.get('Digite seu nome completo:', row.iloc[0])).strip().upper()
        carga = str(row.get('Carga Horária', '40 horas')).strip()

        # TEXTO PARA EQUIPE
        full_segments = [
            ("Certificamos que ", f_reg),
            (nome, f_bold),
            (", participou do ", f_reg),
            ("FESTEX 2025", f_bold),
            (" do Instituto de Ciências Humanas da Universidade de Brasília, na função de ", f_reg),
            ("EQUIPE ORGANIZADORA", f_bold),
            (", com carga horária total de ", f_reg),
            (carga, f_bold),
            (".", f_reg)
        ]

        # Quebra de linha inteligente
        lines, current_line, current_w = [], [], 0
        for text, font in full_segments:
            words = text.split(' ')
            for i, word in enumerate(words):
                word_to_add = word + (" " if i < len(words)-1 else "")
                w = draw.textlength(word_to_add, font=font)
                if current_w + w > LARGURA_MAXIMA:
                    lines.append(current_line)
                    current_line, current_w = [(word_to_add, font)], w
                else:
                    current_line.append((word_to_add, font))
                    current_w += w
        lines.append(current_line)

        # Cálculo da margem para alinhar à esquerda dentro de um bloco centralizado
        margem_esquerda = (L - LARGURA_MAXIMA) / 2

        y_atual = ALTURA_INICIAL
        for line_segments in lines:
            draw_left_aligned_line(draw, line_segments, y_atual, margem_esquerda)
            y_atual += ESPACO_LINHAS

        # Salvar o PDF limpando o nome do arquivo
        nome_limpo = re.sub(r'[^\w\s-]', '', nome).replace(' ', '_')
        caminho_final = os.path.join(pasta_saida, f"Equipe_{nome_limpo}_{index}.pdf").replace('\x00', '')
        img.save(caminho_final)

    # Finalização e Download
    shutil.make_archive('certificados_equipe_final', 'zip', pasta_saida)
    files.download('certificados_equipe_final.zip')
    print("\n🚀 TUDO PRONTO! O download começará automaticamente.")

except Exception as e:
    print(f"❌ Erro crítico: {e}")

1. Selecione a PLANILHA DA EQUIPE:


Saving EquipeAtt(Planilha1) (1).csv to EquipeAtt(Planilha1) (1).csv

2. Selecione o MODELO DA IMAGEM:


Saving modelo.png to modelo.png

⏳ Gerando certificados para a Equipe...


/tmp/ipython-input-3668818169.py:60: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🚀 TUDO PRONTO! O download começará automaticamente.
